# Section 6 — The Classical Fourth-Order Runge–Kutta Method (RK4)

[← RK2 and RK3](05_rk2_rk3.ipynb) · [Module 3](README.md)

The classical RK4 method is the standard fixed-step Runge–Kutta method used in many introductory numerical-analysis courses.

This presentation follows the course lecture notes, **Brin, Chapter 6**, and the classical development in **Burden & Faires**.

### Learning goals
- understand the four slopes used by RK4;
- implement one RK4 step from scratch;
- reproduce the lecture worked example;
- verify fourth-order convergence numerically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## RK4 formula

For a step from \((t_n,w_n)\) of size \(h\),

\[
\begin{aligned}
k_1 &= f(t_n,w_n),\\
k_2 &= f\left(t_n+\frac h2,w_n+\frac h2k_1\right),\\
k_3 &= f\left(t_n+\frac h2,w_n+\frac h2k_2\right),\\
k_4 &= f(t_n+h,w_n+h k_3),\\
w_{n+1} &= w_n+\frac h6(k_1+2k_2+2k_3+k_4).
\end{aligned}
\]

RK4 uses four function evaluations per step and has global error \(O(h^4)\).

In [ ]:
def rk4_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + 0.5*h, y + 0.5*h*k1)
    k3 = f(t + 0.5*h, y + 0.5*h*k2)
    k4 = f(t + h, y + h*k3)
    return y + (h/6)*(k1 + 2*k2 + 2*k3 + k4)

def fixed_step_solver(step_function, f, t0, y0, tf, h):
    n = round((tf-t0)/h)
    t = np.empty(n+1)
    y = np.empty(n+1)
    t[0], y[0] = t0, y0
    for i in range(n):
        y[i+1] = step_function(f, t[i], y[i], h)
        t[i+1] = t[i] + h
    return t, y

## Worked example from the lecture

Consider

\[
y'=-2t y^2,\qquad y(0)=1,
\]

whose exact solution is

\[
y(t)=\frac{1}{1+t^2}.
\]

Take one RK4 step with \(h=0.2\).

In [ ]:
def f_example(t, y):
    return -2*t*y**2

h = 0.2
t = 0.0
y = 1.0

k1 = f_example(t, y)
k2 = f_example(t + h/2, y + (h/2)*k1)
k3 = f_example(t + h/2, y + (h/2)*k2)
k4 = f_example(t + h, y + h*k3)
y1 = y + (h/6)*(k1 + 2*k2 + 2*k3 + k4)

print(f"k1 = {k1:.8f}")
print(f"k2 = {k2:.8f}")
print(f"k3 = {k3:.8f}")
print(f"k4 = {k4:.8f}")
print(f"RK4 y(0.2)  = {y1:.8f}")
print(f"Exact y(0.2)= {1/(1+0.2**2):.8f}")
print(f"Absolute error = {abs(y1-1/(1+0.2**2)):.3e}")

## Compare RK4 with the exact solution over an interval

In [ ]:
t, y = fixed_step_solver(rk4_step, f_example, 0.0, 1.0, 2.0, 0.2)
t_fine = np.linspace(0, 2, 500)
y_exact = 1/(1+t_fine**2)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(t_fine, y_exact, linewidth=2, label="Exact")
ax.plot(t, y, marker="o", label="RK4, h=0.2")
ax.set_xlabel("t")
ax.set_ylabel("y")
ax.set_title("Classical RK4")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

## Fourth-order convergence

Now return to the common test problem

\[
y'=t e^{3t}-2y,\qquad y(0)=0,
\]

so the RK4 error can be compared with the earlier methods.

In [ ]:
def f(t, y):
    return t*np.exp(3*t) - 2*y

def exact(t):
    return np.exp(3*t)*(t/5 - 1/25) + np.exp(-2*t)/25

step_sizes = np.array([0.25, 0.125, 0.0625, 0.03125])
errors = []

for h in step_sizes:
    _, w = fixed_step_solver(rk4_step, f, 0.0, 0.0, 1.0, h)
    errors.append(abs(w[-1] - exact(1.0)))

print(f"{'h':>10s} {'error':>14s} {'observed p':>12s}")
previous = None
for h, err in zip(step_sizes, errors):
    p = "--" if previous is None else f"{np.log2(previous/err):.4f}"
    print(f"{h:10.5f} {err:14.6e} {p:>12s}")
    previous = err

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.loglog(step_sizes, errors, marker="o")
ax.set_xlabel("step size h")
ax.set_ylabel("absolute error at t=1")
ax.set_title("RK4: fourth-order convergence")
ax.grid(True, which="both", alpha=0.25)
plt.show()

## Why RK4 became the workhorse

RK4 is attractive because it combines:
- fourth-order accuracy;
- only function evaluations—no derivatives of \(f\);
- a simple fixed-step implementation.

Its limitation is that a fixed step size does not automatically react to regions where the solution becomes easier or harder to approximate.

The next notebook introduces an **adaptive RK2(3) method**, following Brin's Section 6.5, and then compares all of the methods in this module.

[Next: Adaptive RK23 and comparison →](07_rk23_and_method_comparison.ipynb)